# Sedenion Defect Census: 4096 (Repaired)

The previous notebook used a hand-rolled multiplication that produced
incorrect counts (1680, then 1728). This version ports the authoritative
`cd_sign` from `gororoba_quantum/cd_core.py` verbatim.

Expected result: **1848/4096 = 45.1171875%** defective.


In [ ]:
# cd_sign ported verbatim from gororoba_quantum/cd_core.py
def cd_sign(dim: int, p: int, q: int) -> int:
    sign = 1
    half = dim >> 1
    while half > 0:
        p_hi = p >= half
        q_hi = q >= half
        if not p_hi and not q_hi:
            pass
        elif not p_hi and q_hi:
            qh = q - half
            q = p
            p = qh
        elif p_hi and not q_hi:
            p -= half
            if q != 0:
                sign = -sign
        else:
            qh = q - half
            ph = p - half
            if qh == 0:
                return -sign
            p = qh
            q = ph
        half >>= 1
    return sign

# associator_phase ported verbatim from gororoba_quantum/predict.py
def associator_phase(dim: int, i: int, j: int):
    return [
        cd_sign(dim, i, j) * cd_sign(dim, i ^ j, x) * cd_sign(dim, j, x) * cd_sign(dim, i, j ^ x)
        for x in range(dim)
    ]

print('Authoritative functions loaded.')


## Census: count defective triples
A triple (i,j) is defective if the associator phase is non-trivial
(not all +1). We check all 16×16 = 256 pairs, each with 16 phases = 4096.


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

dim = 16
defective = 0
total = 0
for i in range(dim):
    for j in range(dim):
        phases = associator_phase(dim, i, j)
        for x in range(dim):
            total += 1
            if phases[x] == -1:
                defective += 1

print(f'Defective: {defective}/{total}')
print(f'Percentage: {100*defective/total:.7f}%')
print(f'Expected: 1848/4096 = 45.1171875%')
print(f'MATCH: {defective == 1848 and total == 4096}')


## Cross-check: witness expectation
From the repo's algebra rules: uniform superposition over all 16³ triples
gives witness expectation 1 − 2(1848/4096) = 25/256.


In [ ]:
from fractions import Fraction
expected = Fraction(1) - 2*Fraction(1848, 4096)
print(f'Witness expectation: {expected} = {float(expected):.6f}')
print(f'25/256 = {25/256:.6f}')
print(f'Match: {expected == Fraction(25, 256)}')


## Verdict

The repaired census confirms **1848/4096 = 45.1171875%** defective,
matching the repository's retained theorem exactly.

The previous notebook's hand-rolled implementation is superseded.
